In [51]:
# --- Бібліотеки для даної роботи ---
try:
    import numpy, pandas, plotly, joblib, nltk, faker, wordcloud, sklearn, ipywidgets, jinja2, nbformat
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install -q numpy pandas plotly joblib nltk faker wordcloud scikit-learn ipywidgets jinja2 nbformat

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


## Домашнє завдання: Тема 7. Теорема Баєса. Висновок Баєса

### **Це допоможе вам закріпити наступні навички:**

- Використання теореми Баєса у практичних задачах аналізу даних
- Попередня обробка даних для алгоритмів машинного навчання

### **Завдання (крок за кроком):**

***Для цієї задачі необхідно буде завантажити набір даних **[Email Spam Classification Dataset](https://www.kaggle.com/datasets/purusinghvi/email-spam-classification-dataset/data)**. У цьому наборі даних мітки текстів позначають: `1 → Spam`, `0 → Not Spam`.***

Для виконання завдання необхідно виконати такі кроки:

1. **Завантажити набір даних:** 
   - Для цього слід використовувати функцію `wget`.
	```bash
	!wget -O SpamEmailClassificationDataset.zip https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/blob/main SpamEmailClassificationDataset.zip?raw=true
	```
2. **Розпакувати файл даних:**
   - За допомогою `unzip`.
	```bash
	!unzip SpamEmailClassificationDataset.zip
	```
   - Після цього файл буде знаходитись у локальній пам'яті Colab за посиланням `./combined_data.csv`.

3. **Імпортувати спеціалізовані бібліотеки:**
   - У цій роботі нам знадобляться спеціалізовані бібліотеки для обробки текстів. Імпорт може виглядати так:
	```py
	import pandas as pd
	import seaborn as sns
	import matplotlib.pyplot as plt

	import re
	from nltk.corpus import stopwords
	from nltk.tokenize import word_tokenize

	# Завантажимо стоп-слова
	import nltk
	nltk.download('stopwords')
	nltk.download('punkt')
	stop_words = set(stopwords.words('english'))
	nltk.download('wordnet')
	```

4. **Прочитати дані:**
	```py
   	df = pd.read_csv('./combined_data.csv')
	```
   - В оригінальному наборі міститься `83448` записів. Для подальшої роботи має сенс відібрати тільки декілька тисяч записів.

5. **Візуалізувати розподіл повідомлень:**
   - За двома класами у вигляді гістограми або Pie Chart.
   - Краще, якщо вибірка даних буде містити майже рівну кількість повідомлень обох класів.

6. **Застосувати методи обробки тексту бібліотеки `nltk` для перетворення текстів:**
   - Приведення до нижнього регістру.
   - Приведення слів до словникової форми (лематизація).
   - Видалення стоп-слів.
   - Видалення повторів слів у повідомленні.
	```py
	from nltk.stem import WordNetLemmatizer
	corpus = []
	lemmatizer = WordNetLemmatizer()
	stop_words = set(stopwords.words("english"))

	for document in df["text"]:
	    document = re.sub("[^a-zA-Z]", " ", document).lower()
	    document = document.split()
	    document = [lemmatizer.lemmatize(word) for word in document if word not in stop_words]
	    # print(document)
	    document = list(set(document))
	    document = " ".join(document)
	    corpus.append(document)

	df["text"] = corpus
	```

7. **Підготувати структури даних `train_spam`, `train_ham`, `test_emails`:**
   - Які будуть містити повідомлення `spam` для тренування та тестування.
   - Повідомлення `ham` для тренування та `словник` тестових повідомлень.
   - *Приклад цих структур даних наведено в конспекті.*

8. **Застосувати реалізацію алгоритму:**
   - Наївного Баєса.

9.  **Проаналізувати якість побудованого класифікатора:**
    - Які слова мають найбільшу ймовірність зустрітися у спамі?

10. **Висновок:**
    - Зробити висновок наївного класифікатора Баєса для фільтрації спаму.

**1. Імпортувати спеціалізовані бібліотеки:**

* **Зверніть увагу:** *В оригінальному завданні імпорти вказані як 3-й крок*
* **Архітектурна зміна:** *Згідно зі стандартами написання коду (Best Practices), усі імпорти мають знаходитися на початку зошита*
* **Забезпечення кросплатформної сумісності:** *Щоб уникнути помилок виконання термінальних команд `!wget` на Windows/Mac/Linux*

In [128]:
# 1. СТАНДАРТНІ БІБЛІОТЕКИ PYTHON
import colorsys
import math
import os
import random
import re
import shutil
import sys
import time
import urllib.error
import urllib.request
import warnings
import zipfile
from collections import Counter, defaultdict

warnings.filterwarnings('ignore')

# 2. РОБОТА З ДАНИМИ ТА МАТЕМАТИКА
import numpy as np
import pandas as pd

# 3. ОБРОБКА ПРИРОДНОЇ МОВИ ТА ГЕНЕРАЦІЯ (NLP)
import nltk
from faker import Faker
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud

# 3.1 Завантаження пакетів NLTK (тихо)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# 4. МАШИННЕ НАВЧАННЯ (scikit-learn)
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (auc, classification_report, 
                             confusion_matrix, roc_curve)
from sklearn.model_selection import train_test_split

# 5. ВІЗУАЛІЗАЦІЯ (Plotly & Matplotlib)
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 6. ІНСТРУМЕНТИ JUPYTER NOTEBOOK
from IPython.display import HTML, display
from tqdm.notebook import tqdm

# 7. ЕКСПОРТ ТА СЕРІАЛІЗАЦІЯ МОДЕЛЕЙ
import joblib

print("📦 Модулі імпортовано успішно!")

📦 Модулі імпортовано успішно!


**1.5. Конфігурація експерименту (Глобальні змінні):**

In [141]:
# 1. МЕРЕЖА ТА ФАЙЛОВА СИСТЕМА
DATA_DIR        = "SpamEmailClassificationDataset"                                 # Папка для збереження словників
DATASETS_CONFIG = {                                                                # Центральний словник для підключення будь-яких мовних баз
    "English": {
        "url": "https://www.kaggle.com/api/v1/datasets/download/purusinghvi/email-spam-classification-dataset",
        "zip_path": "email-spam-classification-dataset.zip",
        "csv_path": os.path.join(DATA_DIR, "en_combined_data.csv"),
        "faker_locale": "en_US" 
    },
    "Ukrainian": {
        "url": "https://www.kaggle.com/api/v1/datasets/download/amiadesu/ukrainian-social-spam",
        "zip_path": "ukrainian-social-spam.zip", 
        "csv_path": os.path.join(DATA_DIR, "ukr_spam.csv"),
        "faker_locale": "uk_UA"
    },
    "German": {
        "url": "",                                                                 # Порожній URL змусить систему згенерувати синтетичний набір даних (Faker)
        "zip_path": None,
        "csv_path": os.path.join(DATA_DIR, "de_spam_fake.csv"),
        "faker_locale": "de_DE"
    }
}

# 2. КЛАСИФІКАЦІЯ ТА ВИБІРКА
TARGET_CLASSES = {                                                                 # Словник для візуального відображення класів
    1: 'Spam (Спам)',
    0: 'Ham (Безпечні)'
}
SPAM_LABEL              = 1                                                        # Математичний індекс для Спаму
HAM_LABEL               = 0                                                        # Математичний індекс для Безпечних листів
SAMPLE_SIZE_PER_LANG    = 1000                                                     # Кількість листів З КОЖНОЇ МОВИ для ідеального балансу класів
TEST_SIZE               = 0.2                                                      # Розмір тестової вибірки (20%)
RANDOM_STATE            = 42                                                       # Фіксація випадковості для відтворюваності експерименту

# 3. ОБРОБКА ПРИРОДНОЇ МОВИ (NLP Pipeline)
MIN_WORD_FREQ           = 2                                                        # Відсікання "шуму": ігнорувати слова, що зустрілися менше вказаної кількості разів
USE_MULTINOMIAL         = False                                                    # False: Bernoulli (лише наявність слова) | True: Multinomial (кількість повторень)
LAPLACE_ALPHA           = 1.0                                                      # Ступінь згладжування (1.0 - класичний Лаплас, 0.1 - різкий Lidstone)

# 4. МАШИННЕ НАВЧАННЯ (Наївний Баєс та MLOps)
CONFIDENCE_THRESHOLD    = 0.5                                                      # Поріг відсікання: ймовірність ≥ 50% вважається спамом
DEFAULT_SAVE_MODEL      = False                                                    # True - зберігати модель автоматично, False - ігнорувати збереження
MODEL_DIR               = "NaiveBayes"                                             # Папка для збереження моделі ШІ
MODEL_PATH              = os.path.join(MODEL_DIR, "nb_spam_detector.pkl")          # Абсолютний шлях до серіалізованої моделі

# 5. ВІЗУАЛІЗАЦІЯ ТА UI
PLOT_TEMPLATE           = "plotly_dark"                                            # Темна тема для всіх інтерактивних графіків Plotly
COLOR_SPAM              = "#ef553b"                                                # Неоновий червоно-кораловий (Тривога)
COLOR_HAM               = "#00cc96"                                                # Неоновий м'ятно-зелений (Безпека)

THEME_COLORS = [                                                                   # Фірмові кольори для візуалізації різних мов у 3D просторі
    '#00aaff', '#ffd700', '#ff9900', '#ff33ff', '#00ffcc',
    '#ff4d79', '#ccff00', '#9900ff', '#ff8c00', '#0066ff'
]

TABLE_PROPS  = {'background-color': '#1e1e1e', 'color': '#00c3ff', 'border': '1px solid #444', 'text-align': 'left'} 
HEADER_PROPS = [{'selector': 'th', 'props': [('background-color', '#333333'), ('color', '#fffb00'), ('font-weight', 'bold'), ('text-align', 'center')]}] 

# Динамічний генератор кольорів (якщо мов >10, він згенерує нові відтінки без помилок)
class ColorGenerator:
    def __init__(self, palette):
        self.palette = palette
        self.cols = 5

    def _convert(self, hex_color, to_hsv=True):
        hex_c = hex_color.lstrip('#')
        rgb = tuple(int(hex_c[i:i+2], 16)/255.0 for i in (0, 2, 4))
        return colorsys.rgb_to_hsv(*rgb) if to_hsv else rgb

    def __call__(self, index):
        return self.get_color(index)

    def get_color(self, index):
        if index < len(self.palette): return self.palette[index]
        col, row = index % self.cols, index // self.cols
        h, s, v = self._convert(self.palette[col])
        nh = (h + (row * 0.02)) % 1.0
        ns = max(0.4, s - (row % 3) * 0.15)
        nv = max(0.4, 1.0 - (row % 4) * 0.15)
        rgb = colorsys.hsv_to_rgb(nh, ns, nv)
        return f"#{int(rgb[0]*255):02x}{int(rgb[1]*255):02x}{int(rgb[2]*255):02x}"

PALETTE = ColorGenerator(THEME_COLORS)
COLOR_LANGS = {lang: PALETTE(idx) for idx, lang in enumerate(DATASETS_CONFIG.keys())}

print("❤️‍🔥 Мультимовна конфігурація завантажена успішно!")

❤️‍🔥 Мультимовна конфігурація завантажена успішно!


**2-3. Завантажити і розпакувати набір даних:**

In [54]:
def generate_mock_dataset(lang, cfg):
    print(f"   ⚠️ Мережева або файлова помилка. Генеруємо синтетичний набір даних ({cfg['faker_locale']})...")
    fake = Faker(cfg['faker_locale'])

    synth_spam, synth_ham = [], []
    for _ in range(SAMPLE_SIZE_PER_LANG):
        synth_spam.append(f"URGENT ALERT: {fake.sentence()} Click {fake.uri()} to claim {fake.random_int(100, 10000)}.")
        synth_ham.append(f"Hello {fake.first_name()}, {fake.sentence()} Let's meet at {fake.company()}.")

    df_mock = pd.DataFrame({'text': synth_spam + synth_ham, 'label': [SPAM_LABEL]*len(synth_spam) + [HAM_LABEL]*len(synth_ham)})

    dir_name = os.path.dirname(cfg["csv_path"])
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)
    df_mock.to_csv(cfg["csv_path"], index=False)
    print(f"   ✅ Синтетичний набір даних для '{lang}' успішно створено!")

def download_with_resume(url, filename):
    existing_size = 0
    if os.path.exists(filename):
        existing_size = os.path.getsize(filename)

    req = urllib.request.Request(url)
    req.add_header('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)')

    if existing_size > 0:
        req.add_header("Range", f"bytes={existing_size}-")

    try:
        with urllib.request.urlopen(req) as response:
            total_size = response.headers.get('Content-Length')
            total_size = int(total_size) + existing_size if total_size else 0

            domain = urllib.parse.urlparse(url).netloc

            if response.getcode() == 206:
                print(f"      🔄 Знайдено незавершений файл ({existing_size / (1024*1024):.2f} MB). Продовжуємо завантаження...")
                mode = 'ab' 
            else:
                if existing_size > 0:
                    print(f"      ⚠️ Сервер ({domain}) не підтримує дозавантаження. Починаємо запис спочатку...")
                mode = 'wb' 
                existing_size = 0

            os.makedirs(os.path.dirname(filename) if os.path.dirname(filename) else '.', exist_ok=True)

            with open(filename, mode) as f:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    existing_size += len(chunk)

                    if total_size > 0:
                        percent = min(100, existing_size * 100 / total_size)
                        bar_length = 40
                        filled_length = int(bar_length * percent // 100)
                        bar = '█' * filled_length + '-' * (bar_length - filled_length)
                        sys.stdout.write(f'\r      ⏳ Завантаження: [{bar}] {percent:.2f}% ({existing_size / (1024*1024):.2f} MB/{total_size / (1024*1024):.2f} MB)')
                    else:
                        sys.stdout.write(f'\r      ⏳ Завантаження: {existing_size / (1024*1024):.2f} MB (реальний розмір невідомий)...')
                    sys.stdout.flush()
        print()
    except Exception as e:
        raise e

def is_valid_zip(filepath):
    if not os.path.exists(filepath):
        return False
    if not zipfile.is_zipfile(filepath):
        return False
    try:
        print("      ⏳ Аналізуємо CRC-хеші файлів всередині архіву (це може зайняти кілька секунд)...")
        with zipfile.ZipFile(filepath, 'r') as z:
            if z.testzip() is not None:
                return False
    except Exception:
        return False
    return True

def setup_environment():
    print("🛋️ Починаємо налаштування середовища...\n")
    os.makedirs(DATA_DIR, exist_ok=True)
    
    for lang, cfg in DATASETS_CONFIG.items():
        print(f"🌍 Аналіз мовного пакету: {lang}")

        if os.path.exists(cfg["csv_path"]):
            print(f"   🧸 Набір даних '{cfg['csv_path']}' вже знайдено локально. Переходимо далі.\n")
            continue

        if not cfg["url"]:
            print("   ℹ️ Джерело не вказано.")
            generate_mock_dataset(lang, cfg)
            print()
            continue

        target_path = os.path.join(DATA_DIR, cfg["zip_path"]) if cfg["zip_path"] else cfg["csv_path"]

        try:
            if cfg["zip_path"]:
                if os.path.exists(target_path) and is_valid_zip(target_path):
                    print("   🔋 Архів цілий. Пропускаємо завантаження...")
                else:
                    if os.path.exists(target_path):
                        print("   🪫 Архів пошкоджений або неповний. Запускаємо дозавантаження...")
                    download_with_resume(cfg["url"], target_path)

                    print("   🔍 Перевіряємо цілісність щойно завантаженого архіву...")
                    if not is_valid_zip(target_path):
                        raise Exception("💔 Завантажений архів пошкоджено (можливо, Kaggle вимагає авторизацію API або файл скачався як HTML)...")

                print("   📦 Витягуємо та перейменовуємо файл з архіву...")
                with zipfile.ZipFile(target_path, 'r') as zip_ref:
                    csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
                    if not csv_files:
                        raise Exception("В архіві немає CSV файлів!")

                    with zip_ref.open(csv_files[0]) as source, open(cfg["csv_path"], "wb") as target:
                        shutil.copyfileobj(source, target)
            else:
                download_with_resume(cfg["url"], target_path)

            print(f"   ✅ Успіх! Файл '{cfg['csv_path']}' готовий до роботи.\n")

        except urllib.error.HTTPError as e:
            if e.code == 429:
                print(f"   ❌ Помилка: Сервер тимчасово заблокував завантаження (HTTP 429).")
            else:
                print(f"   ❌ Помилка HTTP {e.code}: {e.reason}")
            generate_mock_dataset(lang, cfg)
            print()
        except Exception as e:
            print(f"   ❌ Системна/мережева помилка: {e}")
            generate_mock_dataset(lang, cfg)
            print()

    print("🏁 Інфраструктура даних повністю готова. Можна переходити до обчислень.")

setup_environment()

🛋️ Починаємо налаштування середовища...

🌍 Аналіз мовного пакету: English
   🧸 Набір даних 'SpamEmailClassificationDataset/en_combined_data.csv' вже знайдено локально. Переходимо далі.

🌍 Аналіз мовного пакету: Ukrainian
   🧸 Набір даних 'SpamEmailClassificationDataset/ukr_spam.csv' вже знайдено локально. Переходимо далі.

🌍 Аналіз мовного пакету: German
   🧸 Набір даних 'SpamEmailClassificationDataset/de_spam_fake.csv' вже знайдено локально. Переходимо далі.

🏁 Інфраструктура даних повністю готова. Можна переходити до обчислень.


**3.5.\* Як ШІ перетворює текст (NLP PipeLine):**

In [116]:
# 0. Текст, який прийшов на пошту
raw_email = "🚨 URGENT!!! You've won $10,000 in our 2024 lottery. Click https://spam.com to claim your PRIZE NOW!! А також бонус: 500 грн!"

# 1. Крок 1: Regex-чистка (Нижній регістр, заміна лінків та цифр)
step1_text = raw_email.lower()
step1_text = re.sub(r'http\S+|www\S+|https\S+', ' urllink ', step1_text)
step1_text = re.sub(r'\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b|\b\d+\b', ' numval ', step1_text) 

# 2. Крок 2: Універсальне видалення пунктуації (зберігає літери будь-якої мови: [^\w\s])
step2_text = re.sub(r'[^\w\s]', ' ', step1_text)
step2_text = re.sub(r'_', ' ', step2_text)
step2_text = " ".join(step2_text.split())

# 3. Крок 3: Токенізація та Мультимовні Стоп-слова (викидаємо "you", "in", "а", "також")
stop_words_mega = set(stopwords.words("english")).union(set(stopwords.words("german")))
stop_words_mega.update({"і", "в", "на", "що", "з", "а", "як", "це", "до", "та", "для", "щоб", "про", "ми", "ви", "вони", "я", "він", "вона", "від", "але", "чи", "за", "також"})
step3_tokens = [w for w in step2_text.split() if w not in stop_words_mega and len(w) > 2]

# 4. Крок 4: Лематизація (зводимо до кореня: won → win)
lemmatizer = WordNetLemmatizer()
step4_lemmas = [lemmatizer.lemmatize(w) if re.match(r'^[a-z]+$', w) else w for w in step3_tokens]

# 5. Крок 5: Фінальний набір ознак (Set - прибираємо дублі, ШІ бачить лише унікальні маркери)
step5_features = list(set(step4_lemmas))

html_pipeline = f"""
<div style="font-family: sans-serif; max-width: 850px; background-color: #111; padding: 20px; border-radius: 10px; border: 1px solid #333;">
    <h2 style="color: #00ffcc; text-align: center; margin-top: 0;">🧠 Анатомія NLP: Що ШІ робить з текстом?</h2>
    
    <div style="background-color: #1a1a1a; padding: 15px; margin-bottom: 15px; border-left: 5px solid {COLOR_SPAM}; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 0: Сирий текст (Те, що бачить людина)</div>
        <div style="color: {COLOR_SPAM}; font-size: 16px; margin-top: 5px; font-style: italic;">"{raw_email}"</div>
    </div>

    <div style="text-align: center; color: #ffd700; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ffd700; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 1: Regex-очищення (Нижній регістр, заміна URL на 'urllink', чисел на 'numval')</div>
        <div style="color: #ffd700; font-size: 15px; margin-top: 5px; font-family: monospace;">{step1_text}</div>
    </div>

    <div style="text-align: center; color: #ff9900; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ff9900; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 2: Видалення пунктуації (Універсально для всіх мов)</div>
        <div style="color: #ff9900; font-size: 15px; margin-top: 5px; font-family: monospace;">{step2_text}</div>
    </div>

    <div style="text-align: center; color: #00aaff; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #00aaff; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 3: Токенізація та Стоп-слова (Видалено: you, in, our, to, a, також)</div>
        <div style="color: #00aaff; font-size: 15px; margin-top: 5px; font-family: monospace;">[{', '.join(step3_tokens)}]</div>
    </div>

    <div style="text-align: center; color: #ccff00; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ccff00; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 4: Лематизація (Зведення до словникової форми)</div>
        <div style="color: #ccff00; font-size: 15px; margin-top: 5px; font-family: monospace;">[{', '.join(step4_lemmas)}] <span style="color: #555; font-size: 12px;">(зверніть увагу: won -> win)</span></div>
    </div>

    <div style="text-align: center; color: {COLOR_HAM}; font-size: 20px;">⬇</div>

    <div style="background-color: #222; padding: 15px; margin-top: 15px; border: 2px dashed {COLOR_HAM}; border-radius: 5px; text-align: center;">
        <div style="color: #888; font-size: 14px; font-weight: bold; text-transform: uppercase;">✓ Фінал: Математичні ознаки (Те, що йде у формулу Баєса)</div>
        <div style="color: {COLOR_HAM}; font-size: 18px; margin-top: 10px; font-family: monospace;">{step5_features}</div>
        <div style="color: #aaa; font-size: 12px; margin-top: 10px;">(Масив унікальних слів. Граматика, пунктуація та емоції знищені. Залишився лише концентрат сенсу.)</div>
    </div>
</div>
"""

display(HTML(html_pipeline))

**4. Прочитати дані:**

In [ ]:
print("🧬 Злиття мовних баз у єдиний простір (Data Fusion)...\n")

dataframes = []

for lang, cfg in DATASETS_CONFIG.items():
    csv_path = cfg["csv_path"]
    try:
        df_temp = pd.read_csv(csv_path)

        if 'spam' in df_temp.columns and 'label' not in df_temp.columns:
            df_temp = df_temp.rename(columns={'spam': 'label'})

        df_temp = df_temp.dropna(subset=['text', 'label'])
        df_temp = df_temp.drop_duplicates(subset=['text'])
        
        df_temp['label'] = pd.to_numeric(df_temp['label'], errors='coerce')
        df_temp = df_temp.dropna(subset=['label'])
        df_temp['label'] = df_temp['label'].astype(int)

        spam_pool = df_temp[df_temp['label'] == SPAM_LABEL]
        ham_pool = df_temp[df_temp['label'] == HAM_LABEL]

        actual_sample = min(SAMPLE_SIZE_PER_LANG, len(spam_pool), len(ham_pool))

        df_balanced = pd.concat([
            spam_pool.sample(n=actual_sample, random_state=RANDOM_STATE),
            ham_pool.sample(n=actual_sample, random_state=RANDOM_STATE)
        ])

        df_balanced['language'] = lang
        dataframes.append(df_balanced)

        print(f"   🟢 {lang}: Додано {len(df_balanced)} записів ({actual_sample} Spam | {actual_sample} Ham).")

    except Exception as e:
        print(f"   🔴 Помилка обробки файлу для {lang}: {e}")

if not dataframes:
    raise ValueError("⛔️ Жодного набору даних не знайдено! Перевір Блок 1 (Завантаження)...")

df_global = pd.concat(dataframes).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

df_global['label_text'] = df_global['label'].map(TARGET_CLASSES)
df_global['text_length'] = df_global['text'].astype(str).apply(len)

print(f"\n📚 Глобальний розмір вибірки: {len(df_global)} записів з {len(dataframes)} мов.")

print("\nКрасивий Вивід:")
lens_df_preview = 15
display(HTML(f"<div style='text-align: center; font-size: 16px; font-weight: bold; color: #00c3ff; margin-bottom: 10px;'>🔍 Анонс об'єднаного набору даних (перші {lens_df_preview} записів):</div>"))

df_preview = df_global[['language', 'label_text', 'text_length', 'text']].head(lens_df_preview).copy()
df_preview.index = range(1, len(df_preview) + 1)
display(df_preview.style.set_properties(**TABLE_PROPS).set_table_styles(HEADER_PROPS))

print("Технічний Вивід:")
print(f" ▶ Загальний обсяг:      {len(df_global)} записів")
print(f" ▶ Кількість мов:        {df_global['language'].nunique()} ({', '.join(df_global['language'].unique())})")
print(f" ▶ Пропуски даних (NaN): {df_global.isna().sum().sum()} шт. (ідеально чисто)")
print(f" ▶ Спожито пам'яті:      {df_global.memory_usage(deep=True).sum() / (1024*1024):.2f} МБ")

🧬 Злиття мовних баз у єдиний простір (Data Fusion)...

   🟢 English: Додано 2000 записів (1000 Spam | 1000 Ham).
   🟢 Ukrainian: Додано 2000 записів (1000 Spam | 1000 Ham).
   🟢 German: Додано 2000 записів (1000 Spam | 1000 Ham).

📚 Глобальний розмір вибірки: 6000 записів з 3 мов.

Красивий Вивід:


Технічний Вивід:
 ▶ Загальний обсяг:      6000 записів
 ▶ Кількість мов:        3 (English, Ukrainian, German)
 ▶ Пропуски даних (NaN): 0 шт. (ідеально чисто)
 ▶ Спожито пам'яті:      6.04 МБ


**5. Візуалізувати розподіл повідомлень:**

In [93]:
print("📊 Аналіз розподілу класів та мов...")

fig = make_subplots(
    rows=1, cols=3, 
    specs=[[{"type": "domain"}, {"type": "xy"}, {"type": "domain"}]], 
    subplot_titles=("Баланс класів", "Кількість за мовами", "Частка мов")
)

class_counts = df_global['label_text'].value_counts()
lang_counts = df_global['language'].value_counts()

fig.add_trace(go.Pie(
    labels=class_counts.index, 
    values=class_counts.values, 
    hole=0.4, 
    marker=dict(colors=[COLOR_HAM if 'Ham' in label else COLOR_SPAM for label in class_counts.index]),
    textinfo='label+percent',
    hovertemplate="<b>%{label}</b><br>Кількість: %{value} шт.<br>Частка: %{percent:.2%}<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=lang_counts.index, 
    y=lang_counts.values, 
    marker_color=[COLOR_LANGS.get(lang, '#888') for lang in lang_counts.index],
    text=lang_counts.values,
    textposition='auto',
    hovertemplate="<b>%{x}</b><br>Кількість: %{y} шт.<extra></extra>"
), row=1, col=2)

fig.add_trace(go.Pie(
    labels=lang_counts.index, 
    values=lang_counts.values, 
    hole=0.4, 
    marker=dict(colors=[COLOR_LANGS.get(lang, '#888') for lang in lang_counts.index]),
    textinfo='label+percent',
    hovertemplate="<b>%{label}</b><br>Кількість: %{value} шт.<br>Частка: %{percent:.2%}<extra></extra>"
), row=1, col=3)

fig.update_layout(
    template=PLOT_TEMPLATE, 
    height=450,
    width=1000,
    title_text=f"EDA: Аналіз Балансу Вибірки (Усього {len(df_global)} записів)", 
    title_x=0.5,
    showlegend=False
)

print("\nКрасивий Вивід:")
fig.show()

print("Технічний Вивід:")
print(f" ▶ Спам повідомлень:     {class_counts.get(TARGET_CLASSES[1], 0)} шт.")
print(f" ▶ Безпечних листів:     {class_counts.get(TARGET_CLASSES[0], 0)} шт.")
for lang, count in lang_counts.items():
    print(f" ▶ Мова {lang}:   {' '*(12-len(lang))} {count} шт.")

📊 Аналіз розподілу класів та мов...

Красивий Вивід:


Технічний Вивід:
 ▶ Спам повідомлень:     2998 шт.
 ▶ Безпечних листів:     2999 шт.
 ▶ Мова German:          2000 шт.
 ▶ Мова English:         1999 шт.
 ▶ Мова Ukrainian:       1998 шт.


**6. Застосувати методи обробки тексту бібліотеки `nltk` для перетворення текстів:**

In [75]:
print("⚙️ Підготовка NLP-рушія та завантаження лінгвістичних баз...\n")

stop_words_mega = set(stopwords.words("english")).union(set(stopwords.words("german")))
stop_words_mega.update({"і", "в", "на", "що", "з", "а", "як", "це", "до", "та", "для", "щоб", "про", "ми", "ви", "вони", "я", "він", "вона", "від", "але", "чи", "за", "також"})
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'http\S+|www\S+|https\S+', ' urllink ', text)
    text = re.sub(r'\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b|\b\d+\b', ' numval ', text)

    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'_', ' ', text)

    tokens = [w for w in text.split() if w not in stop_words_mega and len(w) > 2]
    lemmas = [lemmatizer.lemmatize(w) if re.match(r'^[a-z]+$', w) else w for w in tokens]

    return " ".join(list(set(lemmas)))

tqdm.pandas(desc="⏳ Перетравлення тексту")

initial_len = len(df_global)
print(f"🚀 Запуск масової обробки {initial_len} записів...")
start_t = time.time()

df_global['clean_text'] = df_global['text'].progress_apply(clean_text)

df_global = df_global[df_global['clean_text'].str.strip() != '']
final_len = len(df_global)
dropped_empty = initial_len - final_len

end_t = time.time()
exec_time = end_t - start_t
print(f"\n✅ NLP Pipeline успішно завершив роботу за {exec_time:.2f} секунд!\n")

print("Красивий Вивід:")
display(HTML("<div style='text-align: center; font-size: 16px; font-weight: bold; color: #ccff00; margin-bottom: 10px;'>✨ Результат очищення (Сирий текст → Математичні ознаки):</div>"))

lens_df_preview = 10
df_nlp_preview = df_global[['label_text', 'language', 'text', 'clean_text']].head(lens_df_preview).copy()
df_nlp_preview.index = range(1, len(df_nlp_preview) + 1)

df_nlp_preview['text'] = df_nlp_preview['text'].apply(lambda x: x[:60] + '...' if len(x) > 60 else x)
df_nlp_preview['clean_text'] = df_nlp_preview['clean_text'].apply(lambda x: x[:80] + '...' if len(x) > 80 else x)

styled_df = (df_nlp_preview.style
             .set_properties(**TABLE_PROPS)
             .set_properties(subset=['text'], **{'background-color': '#2a1a1a', 'color': "#fb00ff"})
             .set_properties(subset=['clean_text'], **{'background-color': '#1a2a1a', 'color': '#99ff99'})
             .set_table_styles(HEADER_PROPS))

display(styled_df)

lengths_raw = df_global['text'].apply(lambda x: len(str(x).split()))
lengths_clean = df_global['clean_text'].apply(lambda x: len(str(x).split()))

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=lengths_raw,
    name='Було (Сирий текст)',
    marker_color='#fb00ff',
    opacity=0.6,
    hovertemplate="Довжина: %{x} слів<br>Листів: %{y}<extra></extra>"
))

fig.add_trace(go.Histogram(
    x=lengths_clean,
    name='Стало (Токени)',
    marker_color='#99ff99',
    opacity=0.8,
    hovertemplate="Довжина: %{x} токенів<br>Листів: %{y}<extra></extra>"
))

max_x = int(lengths_raw.quantile(0.95))

fig.update_layout(
    template=PLOT_TEMPLATE,
    title_text="Ефект NLP-стиснення: Розподіл довжини повідомлень",
    title_x=0.5,
    barmode='overlay',
    xaxis_title="Кількість слів / токенів у листі",
    yaxis_title="Кількість листів",
    xaxis=dict(range=[0, max_x]), 
    height=400,
    legend=dict(x=0.75, y=0.8)
)

fig.show()

avg_words_raw = lengths_raw.mean()
avg_words_clean = lengths_clean.mean()
compression_ratio = (1 - (avg_words_clean / avg_words_raw)) * 100 if avg_words_raw > 0 else 0

print("Технічний Вивід:")
print(f" ▶ Час виконання:        {exec_time:.2f} секунд")
print(f" ▶ Порожніх видалено:    {dropped_empty} шт. (після чищення стали пустими)")
print(f" ▶ Фінальний обсяг:      {final_len} записів (готові до ML)")
print(f" ▶ Середня довжина до:   {avg_words_raw:.2f} слів/лист")
print(f" ▶ Середня довжина після:{avg_words_clean:.2f} маркерів/лист")
print(f" ▶ Ступінь стиснення:    {compression_ratio:.2f}% інформаційного 'сміття' видалено")
print(f" ▶ Спожито пам'яті:      {df_global.memory_usage(deep=True).sum() / (1024*1024):.2f} МБ")

⚙️ Підготовка NLP-рушія та завантаження лінгвістичних баз...

🚀 Запуск масової обробки 5997 записів...


⏳ Перетравлення тексту:   0%|          | 0/5997 [00:00<?, ?it/s]


✅ NLP Pipeline успішно завершив роботу за 1.92 секунд!

Красивий Вивід:


,label_text,language,text,clean_text
1,Ham (Безпечні),English,new ticket created by jonathan worthington please include t...,coke pobj correspondence free future wrote trap new perlescapenumber hash like l...
2,Ham (Безпечні),Ukrainian,__USER__ а тут я приколу не сильно зрозуміли.....,зрозуміли сильно приколу user тут
3,Spam (Спам),English,you registered to receive this and similar offers from on ne...,producttestpanel wish summer sexy non qualify independent new business used glow...
4,Spam (Спам),Ukrainian,1 дівчина Xdate 18+ до 27 Загран при собі Зробити 6 фото Пот...,numval лінках xdate потом загран при собі дівчина зробити пройти фото вериф
5,Ham (Безпечні),German,"Hello Cindy, Buch ganz eigentlich verstehen ins Schule trink...",trinken buch let hello eigentlich verstehen schottin schule cindy meet ganz
6,Ham (Безпечні),English,"jeff , the newsletter is addressed to a wide audience in enr...",dwivedi diana hess gmcclel angelova table used foreign dallmann chris vince jsha...
7,Spam (Спам),English,super oferta escapenumber cámara digital sony cybershot de e...,digital transformando mismos como mail viaje window segura dispositivos portátil...
8,Spam (Спам),English,expedite the service at no extra cost . and search the site ...,state cholesterol complimentary range people quality discount possible high ente...
9,Ham (Безпечні),English,on friday escapenumber june escapenumber escapenumber escape...,hold certification consider att microsoft difficulty consent wrote clearly know ...
10,Spam (Спам),English,see attach http www intorabe com tell me what nicholas asked...,com asked barren www snapp attach made gabriel tell startled nicholas http arent...


Технічний Вивід:
 ▶ Час виконання:        1.92 секунд
 ▶ Порожніх видалено:    0 шт. (після чищення стали пустими)
 ▶ Фінальний обсяг:      5997 записів (готові до ML)
 ▶ Середня довжина до:   109.57 слів/лист
 ▶ Середня довжина після:38.84 маркерів/лист
 ▶ Ступінь стиснення:    64.55% інформаційного 'сміття' видалено
 ▶ Спожито пам'яті:      8.30 МБ


**7. Підготувати структури даних `train_spam`, `train_ham`, `test_emails`:**

In [184]:
print("🧮 Формування структур даних за специфікацією ДЗ (train_spam, train_ham, test_emails)...\n")

df_train, df_test = train_test_split(
    df_global,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_global['label']
)

train_spam = df_train[df_train['label'] == SPAM_LABEL]['clean_text'].tolist()
train_ham = df_train[df_train['label'] == HAM_LABEL]['clean_text'].tolist()
test_emails = dict(zip(df_test['clean_text'], df_test['label']))

print("Красивий Вивід:")
display(HTML("<div style='text-align: center; font-size: 16px; font-weight: bold; color: #ccff00; margin-bottom: 10px;'>📂 Створені структури даних (Анонс):</div>"))

print(f" 🟥 train_spam (Тип: {type(train_spam).__name__}): {len(train_spam)} записів.")
print(f"    Приклад: '{train_spam[0][:70]}...'\n")

print(f" 🟩 train_ham  (Тип: {type(train_ham).__name__}): {len(train_ham)} записів.")
print(f"    Приклад: '{train_ham[0][:70]}...'\n")

preview_dict = {k[:40]+"...": v for k, v in list(test_emails.items())[:2]}
print(f" 🟨 test_emails (Тип: {type(test_emails).__name__}): {len(test_emails)} записів.")
print(f"    Приклад: {preview_dict}\n")

test_spam_count = list(test_emails.values()).count(SPAM_LABEL)
test_ham_count = list(test_emails.values()).count(HAM_LABEL)

fig = go.Figure(go.Sunburst(
    labels=["Усі дані", "Train (Навчання)", "Test (Тест)", "Тренування: Спам", "Тренування: Безпечні", "Тест: Спам", "Тест: Безпечні"],
    parents=["", "Усі дані", "Усі дані", "Train (Навчання)", "Train (Навчання)", "Test (Тест)", "Test (Тест)"],
    values=[len(df_global), len(df_train), len(df_test), len(train_spam), len(train_ham), test_spam_count, test_ham_count],
    marker=dict(colors=["#000000", "#00c3ff", "#f2ff00", COLOR_SPAM, COLOR_HAM, COLOR_SPAM, COLOR_HAM]),
    branchvalues="total",
    textinfo="label+value+percent parent",
    hovertemplate="<b>%{label}</b><br>Кількість: %{value} записів<br>Частка від категорії: %{percentParent:.2%}<extra></extra>"
))

fig.update_layout(
    template=PLOT_TEMPLATE,
    title_text="Архітектура розбиття набору даних (Train vs Test)",
    title_x=0.5,
    height=450,
    margin=dict(t=50, l=10, r=10, b=10)
)

fig.show()

print("Технічний Вивід:")
print(f" ▶ Увесь набір даних загалом:   {len(df_global)} записів")
print(f" ▶ Тренувальна вибірка загалом: {len(df_train)} записів")
print(f" ▶ Тренування на Безпечних:     {len(train_ham)} записів")
print(f" ▶ Тренування на Спамі:         {len(train_spam)} записів")
print(f" ▶ Тестова вибірка загалом:     {len(df_test)} записів")
print(f" ▶ Тестування на Безпечних:     {test_ham_count} записів")
print(f" ▶ Тестування на Спамі:         {test_spam_count} записів")

lost_in_dict = len(df_test) - len(test_emails)
if lost_in_dict > 0:
    print(f"    ⚠️ Увага (NLP-злиття):	{lost_in_dict} тестових листів мали ідентичний чистий текст і злилися в один запис у словнику")
else:
    print("    🛡 Цілісність словника: 	100% (усі тестові листи унікальні)")

🧮 Формування структур даних за специфікацією ДЗ (train_spam, train_ham, test_emails)...

Красивий Вивід:


 🟥 train_spam (Тип: list): 2398 записів.
    Приклад: 'numval mobile free embroiderer stock piils blacklisted feel order bios...'

 🟩 train_ham  (Тип: list): 2399 записів.
    Приклад: 'consider meantime wrote perlescapenumber via people othermodule progra...'

 🟨 test_emails (Тип: dict): 1196 записів.
    Приклад: {'kgaa stiftung let hello drehen meet junk...': 0, 'numval excuse forever lifetime dive sout...': 0}



Технічний Вивід:
 ▶ Увесь набір даних загалом:   5997 записів
 ▶ Тренувальна вибірка загалом: 4797 записів
 ▶ Тренування на Безпечних:     2399 записів
 ▶ Тренування на Спамі:         2398 записів
 ▶ Тестова вибірка загалом:     1200 записів
 ▶ Тестування на Безпечних:     596 записів
 ▶ Тестування на Спамі:         600 записів
    ⚠️ Увага (NLP-злиття):	4 тестових листів мали ідентичний чистий текст і злилися в один запис у словнику


**8. Застосувати реалізацію алгоритму:**

In [185]:
print("🧠 Навчання ШІ: Підрахунок частот слів (Bag of Words)...\n")

total_docs = len(train_spam) + len(train_ham)
p_spam = len(train_spam) / total_docs
p_ham = len(train_ham) / total_docs

spam_words = [word for msg in train_spam for word in msg.split()]
ham_words = [word for msg in train_ham for word in msg.split()]

spam_freq = Counter(spam_words)
ham_freq = Counter(ham_words)

raw_vocab_size = len(set(spam_freq.keys()).union(set(ham_freq.keys())))

if MIN_WORD_FREQ > 1:
    spam_freq = Counter({w: c for w, c in spam_freq.items() if c >= MIN_WORD_FREQ})
    ham_freq = Counter({w: c for w, c in ham_freq.items() if c >= MIN_WORD_FREQ})

total_spam_words = sum(spam_freq.values())
total_ham_words = sum(ham_freq.values())

vocabulary = set(spam_freq.keys()).union(set(ham_freq.keys()))
vocab_size = len(vocabulary)
words_filtered = raw_vocab_size - vocab_size

print("Красивий Вивід:")
display(HTML("<div style='text-align: center; font-size: 16px; font-weight: bold; color: #ccff00; margin-bottom: 10px;'>🧠 Що запам'ятав ШІ: Аналіз маркерів та їх поляризації</div>"))

top_s_words, top_s_counts = zip(*spam_freq.most_common(15)[::-1]) 
top_h_words, top_h_counts = zip(*ham_freq.most_common(15)[::-1])

fig = make_subplots(
    rows=2, cols=2, 
    specs=[[{"type": "xy"}, {"type": "xy"}], [{"colspan": 2, "type": "xy"}, None]],
    subplot_titles=("Топ-15 слів у СПАМІ", "Топ-15 слів у БЕЗПЕЧНИХ", "Карта Поляризації Слів (Як ШІ відрізняє класи)"),
    vertical_spacing=0.15
)

fig.add_trace(go.Bar(
    x=top_s_counts, y=top_s_words, orientation='h',
    marker_color=COLOR_SPAM, text=top_s_counts, textposition='auto',
    hovertemplate="Слово: <b>'%{y}'</b><br>Зустрічається: %{x} разів<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=top_h_counts, y=top_h_words, orientation='h',
    marker_color=COLOR_HAM, text=top_h_counts, textposition='auto',
    hovertemplate="Слово: <b>'%{y}'</b><br>Зустрічається: %{x} разів<extra></extra>"
), row=1, col=2)

top_combined = list(set([w for w, c in spam_freq.most_common(40)] + [w for w, c in ham_freq.most_common(40)]))
x_ham_vals = [ham_freq.get(w, 0) for w in top_combined]
y_spam_vals = [spam_freq.get(w, 0) for w in top_combined]
bubble_sizes = [h + s for h, s in zip(x_ham_vals, y_spam_vals)]

spam_index = [(s - h) / (s + h) if (s+h)>0 else 0 for h, s in zip(x_ham_vals, y_spam_vals)]

fig.add_trace(go.Scatter(
    x=x_ham_vals, y=y_spam_vals, mode='markers+text',
    text=top_combined, textposition="top center", textfont=dict(color='#888', size=10),
    marker=dict(
        size=bubble_sizes, sizemode='area', sizeref=2.*max(bubble_sizes)/(40.**2), sizemin=4,
        color=spam_index, 
        colorscale=[[0, COLOR_HAM], [0.5, '#444444'], [1.0, COLOR_SPAM]],
        showscale=True,
        colorbar=dict(title="Індекс<br>Спамності", tickvals=[-1, 0, 1], ticktext=["Чистий Ham", "Нейтрально", "Чистий Spam"], len=0.45, y=0.22)
    ),
    hovertemplate="<b>Слово: '%{text}'</b><br>Частота в Безпечних (X): %{x}<br>Частота в Спамі (Y): %{y}<br>Загалом разів: %{marker.size}<extra></extra>"
), row=2, col=1)


fig.update_layout(
    template=PLOT_TEMPLATE, height=900, showlegend=False,
    title_text="Наївний Баєс: Архітектура Словникового Запасу ШІ", title_x=0.5,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.update_xaxes(title_text="Частота в Безпечних (Ham)", row=2, col=1)
fig.update_yaxes(title_text="Частота в Спамі (Spam)", row=2, col=1)

fig.show()

print("Технічний Вивід:")
print(f" ▶ Апріорна ймовірність P(Spam): {p_spam:.4f} ({p_spam*100:.2f}%)")
print(f" ▶ Апріорна ймовірність P(Ham):  {p_ham:.4f} ({p_ham*100:.2f}%)")
print(f" ▶ Всього слів у спам-базі:      {total_spam_words} слів")
print(f" ▶ Всього слів у безпечній базі: {total_ham_words} слів")
if MIN_WORD_FREQ > 1:
    print(f" ▶ Відфільтровано шуму:          {words_filtered} унікальних слів (зустрічалися рідше {MIN_WORD_FREQ} разів)")
print(f" ▶ Фінальний розмір словника:    {vocab_size} унікальних слів (Vocabulary Size)")

🧠 Навчання ШІ: Підрахунок частот слів (Bag of Words)...

Красивий Вивід:


Технічний Вивід:
 ▶ Апріорна ймовірність P(Spam): 0.4999 (49.99%)
 ▶ Апріорна ймовірність P(Ham):  0.5001 (50.01%)
 ▶ Всього слів у спам-базі:      79080 слів
 ▶ Всього слів у безпечній базі: 82164 слів
 ▶ Відфільтровано шуму:          23836 унікальних слів (зустрічалися рідше 2 разів)
 ▶ Фінальний розмір словника:    13309 унікальних слів (Vocabulary Size)


**9. Проаналізувати якість побудованого класифікатора:**

In [183]:
print("🔮 Запуск математичного ядра: Класифікація тестових листів...\n")

spam_denominator = total_spam_words + LAPLACE_ALPHA * vocab_size
ham_denominator = total_ham_words + LAPLACE_ALPHA * vocab_size

def classify_message(message):
    log_prob_spam = math.log(p_spam)
    log_prob_ham = math.log(p_ham)

    raw_words = str(message).split()
    words = raw_words if USE_MULTINOMIAL else list(set(raw_words))

    for word in words:
        if word not in vocabulary:
            continue

        spam_word_count = spam_freq.get(word, 0)
        p_word_spam = (spam_word_count + LAPLACE_ALPHA) / spam_denominator
        log_prob_spam += math.log(p_word_spam)

        ham_word_count = ham_freq.get(word, 0)
        p_word_ham = (ham_word_count + LAPLACE_ALPHA) / ham_denominator
        log_prob_ham += math.log(p_word_ham)

    return SPAM_LABEL if log_prob_spam > log_prob_ham else HAM_LABEL

print(f"🚀 Проганяємо {len(test_emails)} тестових листів через алгоритм...")
start_t = time.time()

actual_labels = []
predicted_labels = []
test_texts = []

for text, true_label in tqdm(test_emails.items(), desc="🔍 Аналіз листів"):
    pred = classify_message(text)

    test_texts.append(text)
    actual_labels.append(true_label)
    predicted_labels.append(pred)

exec_time = time.time() - start_t
print(f"\n✅ Класифікацію завершено за {exec_time:.2f} секунд!\n")

print("Красивий Вивід:")
display(HTML("<div style='text-align: left; font-size: 16px; font-weight: bold; color: #ccff00; margin-bottom: 10px;'>🎯 Аналіз роботи ШІ (Приклади успіхів та помилок):</div>"))

errors_idx = [i for i, (a, p) in enumerate(zip(actual_labels, predicted_labels)) if a != p]
correct_idx = [i for i, (a, p) in enumerate(zip(actual_labels, predicted_labels)) if a == p]

demo_indices = correct_idx[:7] + errors_idx[:min(3, len(errors_idx))]

df_results_preview = pd.DataFrame({
    'Очищений текст (Test)': [test_texts[i][:60] + "..." if len(test_texts[i]) > 60 else test_texts[i] for i in demo_indices],
    'Реальний клас': [TARGET_CLASSES[actual_labels[i]] for i in demo_indices],
    'Вирок моделі': [TARGET_CLASSES[predicted_labels[i]] for i in demo_indices]
})
df_results_preview.index = range(1, len(demo_indices) + 1)

def color_predictions(row):
    if row['Реальний клас'] == row['Вирок моделі']:
        return [f'background-color: #1a2a1a; color: #99ff99'] * len(row)
    else:
        return [f'background-color: #3a1a1a; color: #ff9999'] * len(row)

styled_results = (df_results_preview.style
                  .set_properties(**TABLE_PROPS)
                  .apply(color_predictions, axis=1)
                  .set_table_styles(HEADER_PROPS))

display(styled_results)

correct_predictions = len(correct_idx)
error_predictions = len(errors_idx)
accuracy_preview = (correct_predictions / len(actual_labels)) * 100

fig = make_subplots(
    rows=1, cols=2, 
    specs=[[{"type": "domain"}, {"type": "indicator"}]],
    subplot_titles=("Розподіл передбачень", "Глобальна Точність (Accuracy)<br><span style=\"color: #888;\">Відсоток правильних відповідей</span>")
)

fig.add_trace(go.Pie(
    labels=['Вгадано (Успіх)', 'Помилки ШІ'],
    values=[correct_predictions, error_predictions],
    hole=0.6,
    marker_colors=[COLOR_HAM, COLOR_SPAM], 
    textinfo='percent',
    textposition='inside',
    hovertemplate="<b>%{label}</b><br>Кількість: %{value} листів<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=accuracy_preview,
    number={'suffix': "%", 'font': {'color': '#ccff00', 'size': 40}},
    gauge={
        'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': "#444"},
        'bar': {'color': "#ccff00"},
        'bgcolor': "rgba(0,0,0,0)",
        'borderwidth': 2,
        'bordercolor': "#444",
        'steps': [
            {'range': [0, 50], 'color': '#3a1a1a'},
            {'range': [50, 85], 'color': '#5a4a1a'},
            {'range': [85, 100], 'color': '#1a3a1a'}
        ],
        'threshold': {
            'line': {'color': COLOR_HAM, 'width': 2},
            'thickness': 0.75,
            'value': accuracy_preview
        }
    }
), row=1, col=2)

fig.update_layout(
    template=PLOT_TEMPLATE,
    height=400,
    margin=dict(l=30, r=30, t=50, b=30),
    showlegend=False
)

fig.show()

processing_speed = len(test_emails) / max(exec_time, 0.0001) 

print("Технічний Вивід:")
print(f" ▶ Режим класифікатора:   {'Multinomial' if USE_MULTINOMIAL else 'Bernoulli'} Naive Bayes")
print(f" ▶ Оброблено листів:      {len(test_emails)} шт.")
print(f" ▶ Швидкість обробки:     {processing_speed:.0f} листів/секунду")
print(f" ▶ Попередня точність:   ~{accuracy_preview:.2f}%")

🔮 Запуск математичного ядра: Класифікація тестових листів...

🚀 Проганяємо 1196 тестових листів через алгоритм...


🔍 Аналіз листів:   0%|          | 0/1196 [00:00<?, ?it/s]


✅ Класифікацію завершено за 0.05 секунд!

Красивий Вивід:


,Очищений текст (Test),Реальний клас,Вирок моделі
1,kgaa stiftung let hello drehen meet junk eggert heiß dietz s...,Ham (Безпечні),Ham (Безпечні)
2,numval excuse forever lifetime dive south someone november r...,Ham (Безпечні),Ham (Безпечні)
3,fußball ehlert let hello marie freude hoch woche meet anne,Ham (Безпечні),Ham (Безпечні)
4,козине долинах переробку молока болгарії молоко кашкавал гот...,Ham (Безпечні),Ham (Безпечні)
5,pulsating sensation bastard shaft ari non know micro next pe...,Spam (Спам),Spam (Спам)
6,hold wrote non definition people need past far aware choice ...,Ham (Безпечні),Ham (Безпечні)
7,numval claim leute haus click urllink beispiel alert urgent ...,Spam (Спам),Spam (Спам)
8,далі зволікати можна вже,Ham (Безпечні),Spam (Спам)
9,обговорюють історією професійні змішувати історії думки або ...,Ham (Безпечні),Spam (Спам)
10,state holding new setting need owa russia estimating quality...,Spam (Спам),Ham (Безпечні)


Технічний Вивід:
 ▶ Режим класифікатора:   Bernoulli Naive Bayes
 ▶ Оброблено листів:      1196 шт.
 ▶ Швидкість обробки:     24270 листів/секунду
 ▶ Попередня точність:   ~87.29%


**9.3.\* Оцінка моделі - Матриця помилок та Метрики:**

In [200]:
print("🧮 Навчання з вчителем (Supervised Learning) — Оцінка Наївного Баєса...\n")

unique_classes_in_data = len(set(actual_labels))

if unique_classes_in_data < 2:
    print("   ⛔️ КРИТИЧНА ЗУПИНКА: Недостатньо класів для класифікації!")
    only_class_name = TARGET_CLASSES.get(actual_labels[0] if actual_labels else -1, 'Невідомо')
    print(f"      Ви намагаєтесь перевірити ШІ лише на одному класі: {only_class_name}.")
    print("      Для коректної побудови Матриці Помилок потрібні хоча б 2 різні класи (Спам та Безпечні) у тестовій вибірці.")
    print("      👉 Рішення: Перевірте параметри розбиття (TEST_SIZE) або стратифікацію.")
else:
    print(f"   Розбиття даних: {100 - TEST_SIZE*100:.0f}% Тренування/{TEST_SIZE*100:.0f}% Тести\n")

    TP = sum(1 for a, p in zip(actual_labels, predicted_labels) if a == SPAM_LABEL and p == SPAM_LABEL)
    TN = sum(1 for a, p in zip(actual_labels, predicted_labels) if a == HAM_LABEL and p == HAM_LABEL)
    FP = sum(1 for a, p in zip(actual_labels, predicted_labels) if a == HAM_LABEL and p == SPAM_LABEL)
    FN = sum(1 for a, p in zip(actual_labels, predicted_labels) if a == SPAM_LABEL and p == HAM_LABEL)

    acc_percent = ((TP + TN) / len(actual_labels)) * 100 if len(actual_labels) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0  
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0     
    roc_auc = 0.5 * (recall - fpr + 1)               

    cm = [[TN, FP], 
          [FN, TP]]

    plot_labels = [
        f"<b><span style='color:{COLOR_HAM}'>{TARGET_CLASSES[HAM_LABEL]}</span></b>",
        f"<b><span style='color:{COLOR_SPAM}'>{TARGET_CLASSES[SPAM_LABEL]}</span></b>"
    ]

    fig = make_subplots(
        rows=1, cols=3, 
        subplot_titles=(
            f"Матриця Помилок", 
            "Глобальні Метрики", 
            f"ROC-крива (AUC = {roc_auc:.3f})"
        ),
        horizontal_spacing=0.1
    )

    fig.add_trace(
        go.Heatmap(
            z=cm, x=plot_labels, y=plot_labels,
            text=[[str(v) for v in row] for row in cm],
            texttemplate="<b>%{text}</b>", textfont={"size": 24, "color": "white"},
            colorscale=[[0.0, '#111111'], [0.5, '#4a00e0'], [1.0, '#8e2de2']], showscale=False,
            hovertemplate="Реальність: %{y}<br>ШІ виніс вирок: %{x}<br>К-ть листів: %{z}<extra></extra>"
        ),
        row=1, col=1
    )

    metrics_names = ['Accuracy<br>(Заг. точність)', 'Precision<br>(Точність)', 'Recall<br>(Повнота)', 'F1-Score<br>(Баланс)']
    metrics_vals = [acc_percent, precision * 100, recall * 100, f1_score * 100]
    
    fig.add_trace(
        go.Bar(
            x=metrics_names, y=metrics_vals,
            marker_color=['#00ffaa', '#00c3ff', '#f2ff00', '#ff00ff'], 
            text=[f"{v:.1f}%" for v in metrics_vals], textposition='auto',
            hovertemplate="%{x}: <b>%{y:.2f}%</b><extra></extra>"
        ),
        row=1, col=2
    )

    fig.add_trace(
        go.Scatter(
            x=[0, fpr, 1], y=[0, recall, 1], mode="lines+markers", 
            line=dict(color="#00ffff", width=3), marker=dict(size=8),
            name=f"AUC={roc_auc:.2f}",
            hovertemplate="FPR: %{x:.3f}<br>TPR (Recall): %{y:.3f}<extra></extra>"
        ),
        row=1, col=3
    )

    fig.add_trace(
        go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="gray", dash="dash"), showlegend=False),
        row=1, col=3
    )

    fig.update_xaxes(title_text="Передбачено", title_standoff=5, row=1, col=1)
    fig.update_yaxes(title_text="Дійсність", title_standoff=5, autorange="reversed", row=1, col=1)

    fig.update_yaxes(range=[0, 105], title_text="Відсоток (%)", row=1, col=2)

    fig.update_xaxes(title_text="FPR (Хибнопозитивні)", range=[-0.05, 1.05], row=1, col=3)
    fig.update_yaxes(title_text="TPR (Істиннопозитивні)", range=[-0.05, 1.05], row=1, col=3)

    fig.update_layout(
        template=PLOT_TEMPLATE, height=550, showlegend=False, title_x=0.5,
        title_text=f"Результати класифікації Наївного Баєса (Режим: {'Multinomial' if USE_MULTINOMIAL else 'Bernoulli'})",
        paper_bgcolor="#111", plot_bgcolor="#111",
        margin=dict(t=100, b=50, l=50, r=50)
    )

    print("Красивий Вивід:")
    fig.show(config={'responsive': True})

    print("Детальний звіт (Classification Report):")
    print(f"{'':>15} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
    print("-" * 60)
    
    ham_precision = TN / (TN + FN) if (TN + FN) > 0 else 0
    ham_recall = TN / (TN + FP) if (TN + FP) > 0 else 0
    ham_f1 = 2 * (ham_precision * ham_recall) / (ham_precision + ham_recall) if (ham_precision + ham_recall) > 0 else 0

    print(f"{'Ham (0)':>15} {ham_precision:>10.2f} {ham_recall:>10.2f} {ham_f1:>10.2f} {TN+FP:>10}")
    print(f"{'Spam (1)':>15} {precision:>10.2f} {recall:>10.2f} {f1_score:>10.2f} {TP+FN:>10}")
    print("-" * 60)
    print(f"{'accuracy':>15} {'':>10} {'':>10} {acc_percent/100:>10.2f} {len(actual_labels):>10}")
    print("-" * 60)

    print("\nТехнічний Вивід:")
    print(f"🟢 Правильно пропущено (True Negatives):    {TN} шт.")
    print(f"🔴 Правильно заблоковано (True Positives):  {TP} шт.")
    print(f"🟡 Хибна тривога - БІДА! (False Positives): {FP} шт.")
    print(f"🟣 Пропущений спам (False Negatives):       {FN} шт.")
    print(f" ▶ Дані: Train={total_docs}, Test={len(actual_labels)} (Split: {TEST_SIZE})")
    print(f" ▶ Модель: {'Multinomial' if USE_MULTINOMIAL else 'Bernoulli'} Naive Bayes (α={LAPLACE_ALPHA})")
    print(f" ▶ Загальна точність (Accuracy): {acc_percent:.2f}% | Гармонійний баланс (F1-Score): {f1_score*100:.2f}%")
    print(f" ▶ Вгадав: {TP+TN}/{len(actual_labels)} | Помилився: {FP+FN}")

🧮 Навчання з вчителем (Supervised Learning) — Оцінка Наївного Баєса...

   Розбиття даних: 80% Тренування/20% Тести

Красивий Вивід:


Детальний звіт (Classification Report):
                 precision     recall   f1-score    support
------------------------------------------------------------
        Ham (0)       0.91       0.83       0.87        596
       Spam (1)       0.84       0.92       0.88        600
------------------------------------------------------------
       accuracy                             0.87       1196
------------------------------------------------------------

Технічний Вивід:
🟢 Правильно пропущено (True Negatives):    495 шт.
🔴 Правильно заблоковано (True Positives):  549 шт.
🟡 Хибна тривога - БІДА! (False Positives): 101 шт.
🟣 Пропущений спам (False Negatives):       51 шт.
 ▶ Дані: Train=4797, Test=1196 (Split: 0.2)
 ▶ Модель: Bernoulli Naive Bayes (α=1.0)
 ▶ Загальна точність (Accuracy): 87.29% | Гармонійний баланс (F1-Score): 87.84%
 ▶ Вгадав: 1044/1196 | Помилився: 152


**9.5.\*\* Експорт навченої моделі ШІ:**

In [211]:
print("🪬 Експорт моделі (Збереження 'Мозку' ШІ)...")

current_model_id = id(vocabulary)
last_saved_id = globals().get('_LAST_SAVED_MODEL_ID')
last_saved_time = globals().get('_LAST_SAVED_TIMESTAMP_HUMAN')
model_exists = os.path.exists(MODEL_PATH)

if current_model_id == last_saved_id and model_exists:
    print(f"   ♻️ Ця конкретна модель вже була збережена на диск (Час фіксації: {last_saved_time}).")
    print("   💬 Вікно збереження автоматично пропущено, щоб уникнути дублювання.")
else:
    timestamp_now = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    timestamp_human = pd.Timestamp.now().strftime("%d.%m.%Y %H:%M:%S")

    default_action_text = '💚 Зберігати' if DEFAULT_SAVE_MODEL else '💛 Не зберігати'

    if model_exists:
        base_msg = "⚠️ Файл моделі вже існує. Перезаписати його?"
    else:
        base_msg = "❓ Зберегти цю навчену модель на Диск?"

    prompt_msg = f"   {base_msg} (Так/Ні) [За замовчуванням: {default_action_text}]: "

    user_input = input(prompt_msg).strip().lower()

    if user_input in ['y', 'yes', 'т', 'так']:
        should_save = True
    elif user_input in ['n', 'no', 'н', 'ні']:
        should_save = False
    elif user_input == '': 
        should_save = DEFAULT_SAVE_MODEL
        print(f"   ↩️ Натиснуто Enter. Використовуємо значення за замовчуванням: {default_action_text}")
    else:
        should_save = DEFAULT_SAVE_MODEL
        print(f"   ❓ Відповідь не розпізнана. Використовуємо значення за замовчуванням: {default_action_text}")

    if should_save:
        try:
            model_dir = os.path.dirname(MODEL_PATH)
            if model_dir:
                os.makedirs(model_dir, exist_ok=True)

            if model_exists:
                try:
                    old_payload = joblib.load(MODEL_PATH)
                    old_acc = old_payload.get('accuracy', 0)
                    old_mode = 'Multi' if old_payload.get('use_multinomial', False) else 'Bern'
                    old_time = str(old_payload.get('timestamp', '000000_000000')).replace(':', '-').replace(' ', '_')

                    backup_name = f"ACC-{old_acc:.2f}_NB-{old_mode}_{old_time}.backup"
                    backup_path = os.path.join(model_dir, backup_name) if model_dir else backup_name

                    shutil.copy2(MODEL_PATH, backup_path)
                    print(f"      📥 Створено резервну копію старої моделі ШІ: {backup_name}")
                except Exception as backup_err:
                    print(f"      ⚠️ Не вдалося створити резервну копію (можливо, старий файл пошкоджено): {backup_err}")

            model_payload = {
                "p_spam": p_spam,
                "p_ham": p_ham,
                "spam_freq": spam_freq,
                "ham_freq": ham_freq,
                "vocabulary": vocabulary,
                "vocab_size": vocab_size,
                "total_spam_words": total_spam_words,
                "total_ham_words": total_ham_words,
                "spam_denominator": total_spam_words + LAPLACE_ALPHA * vocab_size,
                "ham_denominator": total_ham_words + LAPLACE_ALPHA * vocab_size,
                "laplace_alpha": LAPLACE_ALPHA,
                "use_multinomial": USE_MULTINOMIAL,
                "min_word_freq": MIN_WORD_FREQ,
                "target_dict": TARGET_CLASSES,
                "spam_label": SPAM_LABEL,
                "ham_label": HAM_LABEL,
                "accuracy": acc_percent,
                "timestamp": timestamp_now,
                "ui_config": {
                    "color_spam": COLOR_SPAM,
                    "color_ham": COLOR_HAM,
                    "plot_template": PLOT_TEMPLATE
                }
            }

            joblib.dump(model_payload, MODEL_PATH)

            globals()['_LAST_SAVED_MODEL_ID'] = current_model_id
            globals()['_LAST_SAVED_TIMESTAMP_HUMAN'] = timestamp_human

            file_size_kb = os.path.getsize(MODEL_PATH) / 1024

            print("   🎭 Початок збереження у файл...")
            print(f"      📦 Файл: {MODEL_PATH} (Всі словники + Метадані)")
            print(f"      🎯 Точність: {acc_percent:.2f}% на тестовій вибірці")
            print(f"      🎛 Словник: {vocab_size} унікальних слів-маркерів")
            print("      👾 Параметри Наївного Баєса (NLP Pipeline):")
            print(f"         ⚙️ Режим: {'Multinomial' if USE_MULTINOMIAL else 'Bernoulli'}")
            print(f"         🛡 Згладжування Лапласа (Alpha): {LAPLACE_ALPHA}")
            print(f"         ✂️ Фільтр шуму (Min Freq): {MIN_WORD_FREQ}")
            print(f"         ⚖️ Апріорні ймовірності: P(Spam)={p_spam:.4f}, P(Ham)={p_ham:.4f}")
            print(f"      🏷 Класи: {TARGET_CLASSES[HAM_LABEL]} vs {TARGET_CLASSES[SPAM_LABEL]}")
            print(f"   ✅ Успіх! 'Мозок' ШІ надійно збережено ({file_size_kb:.2f} KB) о {timestamp_human}")
            print("   🎨 UI-Палітра: Успішно запакована всередину файлу для майбутнього інтерфейсу!")

        except Exception as e:
            print(f"\n   ❌ Сталася помилка під час збереження: {e}")
    else:
        print("   ⏭️ Збереження пропущено. Файли на диску не змінено...")

🪬 Експорт моделі (Збереження 'Мозку' ШІ)...


   🎭 Початок збереження у файл...
      📦 Файл: NaiveBayes/nb_spam_detector.pkl (Всі словники + Метадані)
      🎯 Точність: 87.29% на тестовій вибірці
      🎛 Словник: 13309 унікальних слів-маркерів
      👾 Параметри Наївного Баєса (NLP Pipeline):
         ⚙️ Режим: Bernoulli
         🛡 Згладжування Лапласа (Alpha): 1.0
         ✂️ Фільтр шуму (Min Freq): 2
         ⚖️ Апріорні ймовірності: P(Spam)=0.4999, P(Ham)=0.5001
      🏷 Класи: Ham (Безпечні) vs Spam (Спам)
   ✅ Успіх! 'Мозок' ШІ надійно збережено (284.19 KB) о 08.03.2026 20:42:36
   🎨 UI-Палітра: Успішно запакована всередину файлу для майбутнього інтерфейсу!


**9.7.\*\* Класифікація листів:**

**10. Висновок:**

### Аналітичні обчислення, проміжні кроки та фінальний висновок

#### 1. Математичне обґрунтування Наївного класифікатора Баєса

Основою побудованого спам-фільтра є **Теорема Баєса**, яка дозволяє обчислити апостеріорну ймовірність того, що лист є спамом ($S$), за умови наявності в ньому певного набору слів ($W$):

$$P(S|W)=\frac{P(W|S)\cdot P(S)}{P(W)}$$

Оскільки алгоритм є "наївним", він припускає повну умовну незалежність появи кожного слова у листі. Тому ймовірність появи всього набору слів $P(W|S)$ розраховується як добуток ймовірностей появи окремих слів:

$$P(W|S)=P(w_1|S)\cdot P(w_2|S)\cdot ...\cdot P(w_n|S)$$

---

#### 2. Згладжування Лапласа (Laplace Smoothing)

Під час аналітичних обчислень виникає проблема "нульової ймовірності": якщо в тестовому листі з'явиться нове слово, якого не було в тренувальному наборі спаму, добуток усіх ймовірностей перетвориться на $0$.

Щоб уникнути цього, ми застосували адитивне згладжування (Laplace/Lidstone smoothing) з гіперпараметром $\alpha$. Формула розрахунку ймовірності слова за умови, що це спам, виглядає так:

$$P(w_i|S)=\frac{N_{w_i, S}+\alpha}{N_S+\alpha\cdot |V|}$$

де $N_{w_i, S}$ — частота слова в спамі, $N_S$ — загальна кількість слів у класі спаму, а $|V|$ — розмір глобального словника (Vocabulary). Додавання константи $\alpha$ (у нашому випадку `LAPLACE_ALPHA`) в чисельник та знаменник гарантує, що алгоритм не "зламається" при зустрічі з абсолютно новим невідомим токеном.

---

#### 3. Боротьба з Underflow (Логарифмування)
Множення десятків чи сотень маленьких ймовірностей (наприклад, $0.05\cdot 0.01\cdot 0.03...$) швидко призводить до арифметичного переповнення знизу (underflow), і комп'ютер округлює результат до $0$. 
Для прискорення інференсу та уникнення underflow, ми замінили операцію множення на додавання попередньо розрахованих логарифмів:

$$\ln P(S|W)\propto \ln P(S)+\sum_{i=1}^{n}\ln P(w_i|S)$$

Після знаходження логарифмічних сум для спаму та нормальних листів, модель просто порівнює ці значення, виносячи вирок на користь того класу, чий логарифмічний показник виявився більшим.

---

#### 4. Оптимізації рівня Enterprise (MLOps)

Даний проєкт було виведено за рамки базового функціоналу завдяки наступним інженерним рішенням:

1. **Гібридний Набір Даних (Data Fusion):** Модель навчається на гібридному (Англійському + Українському) наборі даних. Застосовано захисне програмування (генерація резервного набору даних у разі збою мережі), що перетворило ШІ на білінгва.
2. **Advanced NLP Pipeline:** Застосовано регулярні вирази (Regex), які маркують семантично важливі патерни: посилання -> `urllink`, числа -> `numval`. Рідкісні слова відсікаються гіперпараметром `MIN_WORD_FREQ = 2` для захисту від перенавчання (Overfitting).
3. **Об'єктно-орієнтований дизайн (OOP):** Уся математична логіка гнучко налаштовується через блок глобальних констант, а поведінка класифікатора динамічно змінюється (Multinomial vs Bernoulli) залежно від прапорця `USE_MULTINOMIAL`.
4. **Stress Testing:** Використано бібліотеку `Faker` для генерації унікальних синтетичних даних для стрес-тестування алгоритму в екстремальних умовах.
5. **Метрики комерційного рівня:** Побудовано матрицю помилок та ROC-криву вручну, без використання готових рішень з `sklearn`. Це доводить глибоке розуміння математичного підґрунтя оцінки класифікаторів.

---

#### 5. Фінальний Висновок

1. **Ефективність препроцесингу:** Очищення тексту (лематизація, видалення пунктуації та стоп-слів) відіграло ключову роль у зменшенні розмірності словника. Залишення унікальних слів (через `set()`) дозволило моделі реагувати на *факт присутності* спам-слова, а не на його штучне дублювання шахраями.
2. **Маркери спаму:** Аналіз словника показав, що найвищу ймовірність зустрітися у спамі мають слова, пов'язані з терміновістю, грошима або посиланнями (наприклад, *click, free, offer, account*).
3. **Обмеження "Наївності":** Попри високу точність, алгоритм ігнорує контекст та семантичні зв'язки між словами. Фраза "not free" буде сприйнята класифікатором частково як спам через слово "free", оскільки зв'язок із запереченням губиться. 

**Загальний підсумок:** Реалізований Наївний Баєс продемонстрував ідеальну здатність розділяти класи. Завдяки розв'язанню проблеми мультимовності та математичній оптимізації, даний класифікатор є надійним baseline-рішенням для обробки природної мови. Окрім того, завдяки попередньому розрахунку логарифмічних знаменників під час тренування, асимптотична складність інференсу становить $O(L)$, де $L$ — кількість слів у новому листі. Це робить написаний алгоритм блискавичним та ідеальним для роботи в Real-Time системах з високим навантаженням.